# EUROCONTROL Flight-List Preprocessing & Directed Route-Pair Builder

**Purpose:** transform monthly flight-list Parquet files into validated, dashboard-ready datasets for the Streamlit airline operational performance dashboard.

This notebook is designed to be rerun whenever new monthly files are added. It separates expensive data preparation from interactive dashboard use.

## Outputs

The notebook writes the following files to `processed_data/`:

| Output | Grain | Intended use |
|---|---|---|
| `flights_processed.parquet` | One accepted flight-list record per unique flight ID | Flight explorer and flexible filtering |
| `route_pairs.parquet` | One row per **directed** ADEP → ADES pair | Network map and route ranking |
| `airport_summary.parquet` | One row per airport | Departure/arrival activity |
| `monthly_summary.parquet` | One row per month | Monthly trend charts |
| `duplicate_conflicts.csv` | One row per conflicting flight ID | Manual data-quality review |
| `unmatched_airports.csv` | One row per unmatched airport code | Coordinate matching review |
| `preprocessing_report.json` | Run-level counts and validation outcomes | Reproducibility and audit trail |

## Important interpretation and safety choices

1. **A route is directional.** `EHAM → EGLL` and `EGLL → EHAM` are separate pairs. The pair key is the ordered combination `(adep, ades)`.
2. **Flight IDs are identifiers, not numbers.** They are normalised to strings. This avoids unsafe signed/unsigned integer casting and prevents arithmetic on IDs.
3. **Exact duplicate rows are removed.** If one flight ID has materially different records, that ID is quarantined from the analytical outputs and written to `duplicate_conflicts.csv`. The notebook does not silently pick one route or merge conflicting values.
4. **Airport matching is code-based, not name-based.** Four-character codes are matched to OurAirports `icao_code`; three-character codes can be matched to `iata_code` only when that IATA code is unique in the airport reference file. Ambiguous matches are not guessed.
5. **Coordinates describe airport locations, not flown tracks.** Route lines on a map are straight airport-to-airport network links, not radar trajectories.
6. **`first_seen` to `last_seen` is an observation-span proxy, not confirmed block time.**

> **Scope of validation in this notebook:** structural, key uniqueness, route aggregation reconciliation, coordinate range checks, and explicit exception reporting. It cannot prove that source records themselves are operationally correct; that requires source documentation or independent validation.

## 1. Environment and configuration

Run this notebook from the project root, or change the paths below. By default it expects the same folder structure as the Streamlit project:

```text
airline_flight_list_dashboard/
├── data/
│   ├── airports.csv
│   └── flight_list/
│       ├── flight_list_202501.parquet
│       ├── flight_list_202502.parquet
│       └── ...
├── processed_data/              # created by this notebook
└── flight_data_preprocessing_and_route_pairs.ipynb
```

Install the dependencies once in the environment used by Jupyter:

```bash
python -m pip install --upgrade pandas numpy pyarrow matplotlib
```

**Before running:** make sure all monthly source Parquet files are in `data/flight_list/`. This notebook will process every `*.parquet` file it finds there; it does not manufacture missing months.

In [1]:
from pathlib import Path
import json
import re
import hashlib
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

# ---- Edit these paths if your notebook is stored elsewhere ----
PROJECT_DIR = Path.cwd()
FLIGHT_DIR = PROJECT_DIR / "data" / "flight_list"
AIRPORTS_FILE = PROJECT_DIR / "data" / "airports.csv"
OUTPUT_DIR = PROJECT_DIR / "processed_data"

# Optional: set to True to stop if fewer than 12 distinct months are present.
# Keep False for partial-year processing and testing.
REQUIRE_12_MONTHS = False

# Set True to require every source file to match the expected filename pattern.
STRICT_FILENAME_CHECK = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR.resolve())
print("Flight input directory:", FLIGHT_DIR.resolve())
print("Airport reference:", AIRPORTS_FILE.resolve())
print("Output directory:", OUTPUT_DIR.resolve())


Project directory: C:\Users\Toni Manjani\Desktop\airline_flight_list_dashboard_full_year_map_fixed\airline_flight_list_dashboard
Flight input directory: C:\Users\Toni Manjani\Desktop\airline_flight_list_dashboard_full_year_map_fixed\airline_flight_list_dashboard\data\flight_list
Airport reference: C:\Users\Toni Manjani\Desktop\airline_flight_list_dashboard_full_year_map_fixed\airline_flight_list_dashboard\data\airports.csv
Output directory: C:\Users\Toni Manjani\Desktop\airline_flight_list_dashboard_full_year_map_fixed\airline_flight_list_dashboard\processed_data


## 2. Discover and inventory all monthly source files

The inventory records filenames, file sizes, and inferred month labels. The filename pattern is expected to be similar to `flight_list_202501.parquet`.

This step is deliberately explicit: if only January is present, the notebook will say so rather than imply that a full year was processed.

In [2]:
if not FLIGHT_DIR.exists():
    raise FileNotFoundError(
        f"Flight input folder not found: {FLIGHT_DIR}. "
        "Create data/flight_list and place the monthly Parquet files there."
    )

source_files = sorted(p for p in FLIGHT_DIR.glob("*.parquet") if p.is_file())
if not source_files:
    raise FileNotFoundError(f"No .parquet files found in {FLIGHT_DIR}")

def month_from_filename(path: Path):
    match = re.search(r"(20\d{2})(0[1-9]|1[0-2])", path.stem)
    if not match:
        return pd.NaT
    return pd.Timestamp(f"{match.group(1)}-{match.group(2)}-01")

inventory_rows = []
for path in source_files:
    month = month_from_filename(path)
    inventory_rows.append({
        "file": path.name,
        "size_mb": round(path.stat().st_size / (1024**2), 2),
        "filename_month": month,
        "filename_month_valid": pd.notna(month),
    })

file_inventory = pd.DataFrame(inventory_rows)
display(file_inventory)

distinct_filename_months = file_inventory.loc[
    file_inventory["filename_month_valid"], "filename_month"
].nunique()

print(f"Files discovered: {len(source_files)}")
print(f"Distinct months inferred from filenames: {distinct_filename_months}")
if REQUIRE_12_MONTHS and distinct_filename_months < 12:
    raise ValueError(
        f"Only {distinct_filename_months} distinct filename months found; "
        "12 are required by REQUIRE_12_MONTHS=True."
    )
if distinct_filename_months < 12:
    print("NOTE: This is a partial-year input set. Add missing months before calling it a full-year dataset.")
if STRICT_FILENAME_CHECK and not file_inventory["filename_month_valid"].all():
    raise ValueError("At least one filename does not contain a valid YYYYMM month.")


,file,size_mb,filename_month,filename_month_valid
0,flight_list_202501.parquet,32.35,2025-01-01,True
1,flight_list_202502.parquet,33.37,2025-02-01,True
2,flight_list_202503.parquet,39.15,2025-03-01,True
3,flight_list_202504.parquet,42.82,2025-04-01,True
4,flight_list_202505.parquet,48.10,2025-05-01,True
5,flight_list_202506.parquet,50.00,2025-06-01,True
6,flight_list_202507.parquet,52.69,2025-07-01,True
7,flight_list_202508.parquet,49.82,2025-08-01,True
8,flight_list_202509.parquet,41.96,2025-09-01,True
9,flight_list_202510.parquet,38.39,2025-10-01,True


Files discovered: 12
Distinct months inferred from filenames: 12


## 3. Load the airport reference and build unambiguous code lookups

The OurAirports CSV includes both ICAO and IATA codes. We prefer ICAO when a flight-list code has four characters. IATA matching is used only for three-character codes and only if the code identifies exactly one airport in the reference table.

We also validate coordinate ranges and retain the airport reference attributes useful for map labels and audit.

In [3]:
if not AIRPORTS_FILE.exists():
    raise FileNotFoundError(
        f"Airport reference file not found: {AIRPORTS_FILE}. "
        "Place airports.csv in the project's data folder."
    )

airports_raw = pd.read_csv(AIRPORTS_FILE, low_memory=False)
required_airport_cols = {"latitude_deg", "longitude_deg"}
missing = required_airport_cols - set(airports_raw.columns)
if missing:
    raise ValueError(f"Airport reference is missing required columns: {sorted(missing)}")

for col in ["icao_code", "iata_code", "ident", "name", "type", "municipality", "iso_country"]:
    if col not in airports_raw.columns:
        airports_raw[col] = pd.NA
    airports_raw[col] = airports_raw[col].astype("string").str.strip().str.upper()
    airports_raw[col] = airports_raw[col].replace("", pd.NA)

airports_raw["latitude_deg"] = pd.to_numeric(airports_raw["latitude_deg"], errors="coerce")
airports_raw["longitude_deg"] = pd.to_numeric(airports_raw["longitude_deg"], errors="coerce")

valid_coord = (
    airports_raw["latitude_deg"].between(-90, 90)
    & airports_raw["longitude_deg"].between(-180, 180)
)
airports_ref = airports_raw.loc[valid_coord].copy()

# Restrict the map lookup to airports with a usable ICAO or IATA identifier.
airports_ref["icao_code"] = airports_ref["icao_code"].replace("", pd.NA)
airports_ref["iata_code"] = airports_ref["iata_code"].replace("", pd.NA)

# Build a deterministic ICAO lookup. If a code has multiple rows, keep it only
# when all candidate coordinates are identical; otherwise mark it ambiguous.
def build_code_lookup(frame, code_col):
    records = []
    for code, group in frame.dropna(subset=[code_col]).groupby(code_col):
        coord_pairs = group[["latitude_deg", "longitude_deg"]].drop_duplicates()
        if len(coord_pairs) == 1:
            row = group.sort_values(["type", "name"], na_position="last").iloc[0]
            records.append({
                "code": code,
                "latitude_deg": float(row["latitude_deg"]),
                "longitude_deg": float(row["longitude_deg"]),
                "airport_name": row["name"],
                "municipality": row["municipality"],
                "iso_country": row["iso_country"],
                "reference_ident": row["ident"],
                "candidate_rows": len(group),
                "ambiguous": False,
            })
        else:
            records.append({
                "code": code, "latitude_deg": np.nan, "longitude_deg": np.nan,
                "airport_name": pd.NA, "municipality": pd.NA,
                "iso_country": pd.NA, "reference_ident": pd.NA,
                "candidate_rows": len(group), "ambiguous": True,
            })
    return pd.DataFrame(records)

icao_lookup = build_code_lookup(airports_ref, "icao_code")
iata_lookup = build_code_lookup(airports_ref, "iata_code")

# For IATA, require a single source row for the code, not just coincident coordinates.
iata_counts = airports_ref.dropna(subset=["iata_code"]).groupby("iata_code").size()
iata_lookup["ambiguous"] = iata_lookup["code"].map(iata_counts).fillna(0).gt(1)

print("Airport reference rows:", len(airports_raw))
print("Rows with valid coordinates:", len(airports_ref))
print("ICAO lookup codes:", len(icao_lookup))
print("IATA lookup codes:", len(iata_lookup))
print("Ambiguous ICAO codes:", int(icao_lookup["ambiguous"].sum()))
print("Ambiguous IATA codes:", int(iata_lookup["ambiguous"].sum()))


Airport reference rows: 86337
Rows with valid coordinates: 86337
ICAO lookup codes: 10666
IATA lookup codes: 9053
Ambiguous ICAO codes: 0
Ambiguous IATA codes: 0


## 4. Define normalisation and timestamp parsing

Column names are normalised to lowercase canonical names. Codes are trimmed and uppercased. IDs are kept as strings.

The date field `dof` (date of flight) is preferred for assigning a calendar month. When it is absent or invalid for a row, the date portion of `first_seen` is used as a documented fallback. The original raw fields are retained.

In [4]:
ALIASES = {
    "id": ["id"],
    "icao_operator": ["icao_operator"],
    "icao24": ["icao24"],
    "flt_id": ["flt_id"],
    "dof": ["dof"],
    "adep": ["adep"],
    "ades": ["ades"],
    "registration": ["registration"],
    "model": ["model"],
    "typecode": ["typecode"],
    "icao_aircraft_class": ["icao_aircraft_class"],
    "first_seen": ["first_seen"],
    "last_seen": ["last_seen"],
    "callsign": ["callsign"],
    "version": ["version"],
    "unix_time": ["unix_time"],
}

CODE_COLUMNS = [
    "icao_operator", "icao24", "adep", "ades", "typecode",
    "icao_aircraft_class", "registration", "model", "callsign", "flt_id"
]

def normalise_columns(frame):
    lookup = {str(c).strip().lower(): c for c in frame.columns}
    rename = {}
    for canonical, candidates in ALIASES.items():
        for candidate in candidates:
            if candidate in lookup:
                rename[lookup[candidate]] = canonical
                break
    return frame.rename(columns=rename)

def normalise_id(series):
    # String conversion avoids signed/unsigned integer casting failures.
    out = series.astype("string").str.strip()
    return out.replace("", pd.NA)

def parse_mixed_timestamp(series):
    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(series, errors="coerce")
        finite = numeric.dropna()
        if finite.empty:
            return pd.to_datetime(series, errors="coerce", utc=True)
        unit = "ms" if finite.abs().median() > 1e11 else "s"
        return pd.to_datetime(numeric, unit=unit, errors="coerce", utc=True)
    return pd.to_datetime(series, errors="coerce", utc=True)

def parse_dof(series):
    # DOF commonly appears as YYYYMMDD. Also accept already date-like values.
    if pd.api.types.is_numeric_dtype(series):
        text = pd.to_numeric(series, errors="coerce").round().astype("Int64").astype("string")
        return pd.to_datetime(text, format="%Y%m%d", errors="coerce", utc=True)
    text = series.astype("string").str.strip()
    compact = pd.to_datetime(text, format="%Y%m%d", errors="coerce", utc=True)
    general = pd.to_datetime(text, errors="coerce", utc=True)
    return compact.fillna(general)


## 5. Read and standardise every source file

Each row is tagged with its source filename. Missing required columns cause a clear error. A source file is not silently skipped if it cannot be read.

At this stage we do not yet remove conflicting duplicate IDs; that is handled as a separate, auditable step.

In [5]:
frames = []
file_read_audit = []

for path in source_files:
    print(f"Reading {path.name} ...")
    raw = pd.read_parquet(path)  # requires pyarrow or fastparquet
    frame = normalise_columns(raw)

    if "id" not in frame.columns:
        raise ValueError(f"{path.name}: required flight identifier column 'id' is missing.")
    if "adep" not in frame.columns or "ades" not in frame.columns:
        raise ValueError(f"{path.name}: required route columns ADEP and ADES are missing.")

    frame["id"] = normalise_id(frame["id"])
    frame = frame.dropna(subset=["id"]).copy()

    for col in CODE_COLUMNS:
        if col in frame.columns:
            frame[col] = frame[col].astype("string").str.strip()
            frame[col] = frame[col].replace("", pd.NA)
            if col in ["icao_operator", "icao24", "adep", "ades", "typecode", "icao_aircraft_class"]:
                frame[col] = frame[col].str.upper()

    if "dof" in frame.columns:
        frame["dof_dt"] = parse_dof(frame["dof"])
    else:
        frame["dof_dt"] = pd.NaT

    for col in ["first_seen", "last_seen"]:
        if col in frame.columns:
            frame[col + "_dt"] = parse_mixed_timestamp(frame[col])
        else:
            frame[col + "_dt"] = pd.NaT

    # Flight date: DOF first, then first_seen date as row-level fallback.
    frame["flight_datetime_utc"] = frame["dof_dt"]
    fallback = frame["flight_datetime_utc"].isna() & frame["first_seen_dt"].notna()
    frame.loc[fallback, "flight_datetime_utc"] = frame.loc[fallback, "first_seen_dt"]
    frame["flight_date"] = frame["flight_datetime_utc"].dt.date
    frame["flight_month"] = frame["flight_datetime_utc"].dt.to_period("M").astype("string")

    frame["observation_span_min"] = (
        (frame["last_seen_dt"] - frame["first_seen_dt"]).dt.total_seconds() / 60
    )
    # Negative or zero observation spans are not plausible elapsed spans.
    frame.loc[frame["observation_span_min"] <= 0, "observation_span_min"] = np.nan

    frame["source_file"] = path.name
    frame["source_filename_month"] = month_from_filename(path)
    frames.append(frame)

    file_read_audit.append({
        "file": path.name,
        "rows_after_missing_id_removal": len(frame),
        "unique_ids_in_file": frame["id"].nunique(),
        "missing_id_rows": int(raw.shape[0] - len(frame)),
        "columns": len(raw.columns),
    })

all_raw_standardised = pd.concat(frames, ignore_index=True, sort=False)
file_read_audit = pd.DataFrame(file_read_audit)
display(file_read_audit)

print("Combined rows before cross-file duplicate handling:", len(all_raw_standardised))
print("Distinct non-missing flight IDs:", all_raw_standardised["id"].nunique())
print("Rows missing ADEP:", int(all_raw_standardised["adep"].isna().sum()))
print("Rows missing ADES:", int(all_raw_standardised["ades"].isna().sum()))


Reading flight_list_202501.parquet ...


C:\Users\Toni Manjani\AppData\Local\Temp\ipykernel_32120\1872039907.py:40: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  frame["flight_month"] = frame["flight_datetime_utc"].dt.to_period("M").astype("string")


Reading flight_list_202502.parquet ...


C:\Users\Toni Manjani\AppData\Local\Temp\ipykernel_32120\1872039907.py:40: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  frame["flight_month"] = frame["flight_datetime_utc"].dt.to_period("M").astype("string")


Reading flight_list_202503.parquet ...


C:\Users\Toni Manjani\AppData\Local\Temp\ipykernel_32120\1872039907.py:40: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  frame["flight_month"] = frame["flight_datetime_utc"].dt.to_period("M").astype("string")


... (16 outputs left out)

Reading flight_list_202512.parquet ...


C:\Users\Toni Manjani\AppData\Local\Temp\ipykernel_32120\1872039907.py:40: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  frame["flight_month"] = frame["flight_datetime_utc"].dt.to_period("M").astype("string")


,file,rows_after_missing_id_removal,unique_ids_in_file,missing_id_rows,columns
0,flight_list_202501.parquet,1036088,1035898,0,17
1,flight_list_202502.parquet,1066341,1066341,0,17
2,flight_list_202503.parquet,1236032,1235632,0,17
3,flight_list_202504.parquet,1351975,1351975,0,17
4,flight_list_202505.parquet,1518167,1518167,0,17
5,flight_list_202506.parquet,1574471,1574471,0,17
6,flight_list_202507.parquet,1662355,1662355,0,17
7,flight_list_202508.parquet,1573717,1573717,0,17
8,flight_list_202509.parquet,1536783,1535719,0,16
9,flight_list_202510.parquet,1408966,1408966,0,16


Combined rows before cross-file duplicate handling: 16267093
Distinct non-missing flight IDs: 16264265
Rows missing ADEP: 7731010
Rows missing ADES: 7095398


## 6. Deduplicate safely and quarantine conflicting flight IDs

The same flight can appear in overlapping monthly extracts. The following policy avoids double-counting without inventing a route:

- Completely identical rows are collapsed.
- If an ID still appears more than once and its records differ, all records for that ID are excluded from the clean analytical table and saved to a conflict audit.
- The conflict audit records how many source rows exist and which fields differ.
- We do **not** choose a route based on arbitrary file order.

This is intentionally conservative. If conflicting IDs are found, inspect the audit and source documentation before deciding whether a more specific resolution rule is justified.

In [6]:
# Drop byte-for-byte/logically identical rows across all fields.
before_exact = len(all_raw_standardised)
dedup_exact = all_raw_standardised.drop_duplicates().copy()
exact_duplicates_removed = before_exact - len(dedup_exact)

# Determine whether repeated IDs have more than one distinct record.
duplicate_mask = dedup_exact["id"].duplicated(keep=False)
duplicate_rows = dedup_exact.loc[duplicate_mask].copy()

conflict_ids = []
conflict_audit_rows = []

for fid, group in duplicate_rows.groupby("id", sort=False, dropna=False):
    differing_fields = []
    for col in dedup_exact.columns:
        if col == "id":
            continue
        vals = group[col].astype("string").fillna("<NULL>").unique()
        if len(vals) > 1:
            differing_fields.append(col)
    if differing_fields:
        conflict_ids.append(fid)
        conflict_audit_rows.append({
            "id": str(fid),
            "source_rows_after_exact_dedup": len(group),
            "source_files": "; ".join(sorted(group["source_file"].dropna().astype(str).unique())),
            "conflicting_fields": "; ".join(differing_fields),
            "adep_values": "; ".join(sorted(group["adep"].dropna().astype(str).unique())),
            "ades_values": "; ".join(sorted(group["ades"].dropna().astype(str).unique())),
        })

duplicate_conflicts = pd.DataFrame(conflict_audit_rows)

# Keep only IDs represented by one distinct row after exact duplicate removal.
conflict_id_set = set(conflict_ids)
flights = dedup_exact.loc[~dedup_exact["id"].isin(conflict_id_set)].copy()

# Defensive uniqueness assertion: every accepted ID must occur once.
assert flights["id"].is_unique, "Accepted flight IDs are not unique after deduplication."

print("Rows before exact duplicate removal:", before_exact)
print("Exact duplicate rows removed:", exact_duplicates_removed)
print("Conflicting flight IDs quarantined:", len(conflict_id_set))
print("Rows in accepted flight table:", len(flights))
print("Accepted unique flight IDs:", flights["id"].nunique())
display(duplicate_conflicts.head(20))


Rows before exact duplicate removal: 16267093
Exact duplicate rows removed: 2828
Conflicting flight IDs quarantined: 0
Rows in accepted flight table: 16264265
Accepted unique flight IDs: 16264265


""


## 7. Match ADEP and ADES to airport coordinates

ADEP and ADES are treated as airport codes. For each endpoint:

- Four-character values are matched against `icao_code`.
- Three-character values are matched against a unique `iata_code`.
- Codes with no match or ambiguous matches remain unmatched and are listed in `unmatched_airports.csv`.
- We do not infer a location from airport names, country, or nearest-neighbour coordinates.

The source code and match method are retained so map coverage can be audited.

In [8]:
icao_map = icao_lookup.set_index("code").to_dict("index")
iata_map = iata_lookup.set_index("code").to_dict("index")

def lookup_airport(code):
    if pd.isna(code):
        return {
            "latitude_deg": np.nan, "longitude_deg": np.nan,
            "airport_name": pd.NA, "municipality": pd.NA,
            "iso_country": pd.NA, "match_method": "missing_code",
            "match_status": "unmatched",
        }
    code = str(code).strip().upper()
    if len(code) == 4 and code in icao_map:
        item = icao_map[code]
        if not item["ambiguous"]:
            return {**item, "match_method": "icao_code", "match_status": "matched"}
        return {
            "latitude_deg": np.nan, "longitude_deg": np.nan,
            "airport_name": pd.NA, "municipality": pd.NA, "iso_country": pd.NA,
            "match_method": "icao_code", "match_status": "ambiguous",
        }
    if len(code) == 3 and code in iata_map:
        item = iata_map[code]
        if not item["ambiguous"]:
            return {**item, "match_method": "iata_code", "match_status": "matched"}
        return {
            "latitude_deg": np.nan, "longitude_deg": np.nan,
            "airport_name": pd.NA, "municipality": pd.NA, "iso_country": pd.NA,
            "match_method": "iata_code", "match_status": "ambiguous",
        }
    return {
        "latitude_deg": np.nan, "longitude_deg": np.nan,
        "airport_name": pd.NA, "municipality": pd.NA, "iso_country": pd.NA,
        "match_method": "no_reference_match", "match_status": "unmatched",
    }

# Unique code dimension avoids repeated lookups for every flight.
all_codes = pd.Series(pd.concat([flights["adep"], flights["ades"]], ignore_index=True).dropna().unique())
airport_match_rows = []
for code in all_codes:
    airport_match_rows.append({"airport_code": code, **lookup_airport(code)})
airport_match = pd.DataFrame(airport_match_rows)

# Attach origin and destination metadata independently.
airport_meta = airport_match.set_index("airport_code")
for role, code_col in [("origin", "adep"), ("destination", "ades")]:
    flights[f"{role}_latitude"] = flights[code_col].map(airport_meta["latitude_deg"])
    flights[f"{role}_longitude"] = flights[code_col].map(airport_meta["longitude_deg"])
    flights[f"{role}_name"] = flights[code_col].map(airport_meta["airport_name"])
    flights[f"{role}_match_status"] = flights[code_col].map(airport_meta["match_status"])
    flights[f"{role}_match_method"] = flights[code_col].map(airport_meta["match_method"])

# Build one row per distinct code that could not be mapped.
unmatched_codes = airport_match.loc[
    airport_match["match_status"] != "matched"
].copy().sort_values("airport_code")

print("Unique airport codes used by flights:", len(airport_match))
print("Matched codes:", int((airport_match["match_status"] == "matched").sum()))
print("Unmatched or ambiguous codes:", len(unmatched_codes))
display(unmatched_codes.head(30))


Unique airport codes used by flights: 1115
Matched codes: 925
Unmatched or ambiguous codes: 190


,airport_code,latitude_deg,longitude_deg,airport_name,municipality,iso_country,reference_ident,candidate_rows,ambiguous,match_method,match_status
876,AZ-0013,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,no_reference_match,unmatched
813,AZ-0021,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,no_reference_match,unmatched
777,BG-JAM,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,no_reference_match,unmatched
1064,BY-0001,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,no_reference_match,unmatched
1008,BY-0003,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,no_reference_match,unmatched
886,BY-0006,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,no_reference_match,unmatched
624,CY-0002,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,no_reference_match,unmatched
573,EBBL,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,no_reference_match,unmatched
598,EBFS,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,no_reference_match,unmatched
169,EDTD,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,no_reference_match,unmatched


## 8. Create the directed flight-pair dataset

The route-pair output is aggregated by the ordered pair `(adep, ades)`. This preserves direction and prevents opposite directions from being combined.

For each pair we calculate:
- number of accepted flight records;
- unique aircraft and operators when those columns exist;
- median observation span (clearly labelled as a proxy);
- origin/destination airport names and coordinates;
- whether both endpoints are mapped.

A route with an unmapped endpoint is still retained in the route table for traffic analysis; it simply cannot be drawn accurately on the coordinate map.

In [9]:
route_source = flights.dropna(subset=["adep", "ades"]).copy()
route_source = route_source.loc[
    route_source["adep"].ne("") & route_source["ades"].ne("")
].copy()

agg_spec = {
    "flight_count": ("id", "nunique"),
    "median_observation_span_min": ("observation_span_min", "median"),
}
if "icao24" in route_source.columns:
    agg_spec["unique_aircraft"] = ("icao24", "nunique")
if "icao_operator" in route_source.columns:
    agg_spec["unique_operators"] = ("icao_operator", "nunique")
if "typecode" in route_source.columns:
    agg_spec["unique_typecodes"] = ("typecode", "nunique")

route_pairs = (
    route_source.groupby(["adep", "ades"], dropna=False)
    .agg(**agg_spec)
    .reset_index()
)

# Join origin and destination coordinate dimensions.
origin_dim = airport_match.rename(columns={
    "airport_code": "adep",
    "latitude_deg": "origin_latitude",
    "longitude_deg": "origin_longitude",
    "airport_name": "origin_airport_name",
    "municipality": "origin_municipality",
    "iso_country": "origin_country",
    "match_status": "origin_match_status",
    "match_method": "origin_match_method",
})[["adep", "origin_latitude", "origin_longitude", "origin_airport_name",
   "origin_municipality", "origin_country", "origin_match_status", "origin_match_method"]]

destination_dim = airport_match.rename(columns={
    "airport_code": "ades",
    "latitude_deg": "destination_latitude",
    "longitude_deg": "destination_longitude",
    "airport_name": "destination_airport_name",
    "municipality": "destination_municipality",
    "iso_country": "destination_country",
    "match_status": "destination_match_status",
    "match_method": "destination_match_method",
})[["ades", "destination_latitude", "destination_longitude", "destination_airport_name",
   "destination_municipality", "destination_country", "destination_match_status",
   "destination_match_method"]]

route_pairs = route_pairs.merge(origin_dim, on="adep", how="left", validate="many_to_one")
route_pairs = route_pairs.merge(destination_dim, on="ades", how="left", validate="many_to_one")

route_pairs["route"] = route_pairs["adep"].astype(str) + " → " + route_pairs["ades"].astype(str)
route_pairs["route_key"] = route_pairs["adep"].astype(str) + "|" + route_pairs["ades"].astype(str)
route_pairs["is_mapped"] = (
    route_pairs["origin_latitude"].notna()
    & route_pairs["origin_longitude"].notna()
    & route_pairs["destination_latitude"].notna()
    & route_pairs["destination_longitude"].notna()
)
route_pairs = route_pairs.sort_values(
    ["flight_count", "adep", "ades"], ascending=[False, True, True]
).reset_index(drop=True)

print("Accepted flights with both ADEP and ADES:", len(route_source))
print("Unique directed route pairs:", len(route_pairs))
print("Mapped directed route pairs:", int(route_pairs["is_mapped"].sum()))
display(route_pairs.head(20))


Accepted flights with both ADEP and ADES: 5578760
Unique directed route pairs: 86928
Mapped directed route pairs: 78462


,adep,ades,flight_count,median_observation_span_min,unique_aircraft,unique_operators,unique_typecodes,origin_latitude,origin_longitude,origin_airport_name,...,destination_latitude,destination_longitude,destination_airport_name,destination_municipality,destination_country,destination_match_status,destination_match_method,route,route_key,is_mapped
0,ENVA,ENGM,6899,44.250000,258,18,37,63.457802,10.924000,"TRONDHEIM AIRPORT, VÆRNES",...,60.193901,11.100400,OSLO-GARDERMOEN INTERNATIONAL AIRPORT,OSLO (GARDERMOEN),NO,matched,icao_code,ENVA → ENGM,ENVA|ENGM,True
1,ENBR,ENGM,6711,39.416667,243,15,34,60.293400,5.218140,"BERGEN AIRPORT, FLESLAND",...,60.193901,11.100400,OSLO-GARDERMOEN INTERNATIONAL AIRPORT,OSLO (GARDERMOEN),NO,matched,icao_code,ENBR → ENGM,ENBR|ENGM,True
2,EKCH,ENGM,6066,53.500000,313,20,38,55.617901,12.656000,COPENHAGEN KASTRUP AIRPORT,...,60.193901,11.100400,OSLO-GARDERMOEN INTERNATIONAL AIRPORT,OSLO (GARDERMOEN),NO,matched,icao_code,EKCH → ENGM,EKCH|ENGM,True
3,EIDW,EGLL,5757,62.083333,187,8,8,53.428713,-6.262121,DUBLIN AIRPORT,...,51.470748,-0.459909,LONDON HEATHROW AIRPORT,LONDON,GB,matched,icao_code,EIDW → EGLL,EIDW|EGLL,True
4,EGLL,EIDW,5726,51.750000,188,7,10,51.470748,-0.459909,LONDON HEATHROW AIRPORT,...,53.428713,-6.262121,DUBLIN AIRPORT,DUBLIN,IE,matched,icao_code,EGLL → EIDW,EGLL|EIDW,True
5,LPPT,LEMD,5667,56.166667,479,36,53,38.781300,-9.135920,LISBON HUMBERTO DELGADO AIRPORT,...,40.493407,-3.572249,ADOLFO SUÁREZ MADRID–BARAJAS AIRPORT,MADRID,ES,matched,icao_code,LPPT → LEMD,LPPT|LEMD,True
6,ESSA,EKCH,5609,58.166667,274,27,37,59.648490,17.928829,STOCKHOLM-ARLANDA AIRPORT,...,55.617901,12.656000,COPENHAGEN KASTRUP AIRPORT,COPENHAGEN,DK,matched,icao_code,ESSA → EKCH,ESSA|EKCH,True
7,LEBL,LEMD,5604,61.500000,497,50,62,41.297100,2.078460,JOSEP TARRADELLAS BARCELONA-EL PRAT AIRPORT,...,40.493407,-3.572249,ADOLFO SUÁREZ MADRID–BARAJAS AIRPORT,MADRID,ES,matched,icao_code,LEBL → LEMD,LEBL|LEMD,True
8,ENZV,ENGM,5582,41.250000,211,16,25,58.876701,5.637780,"STAVANGER AIRPORT, SOLA",...,60.193901,11.100400,OSLO-GARDERMOEN INTERNATIONAL AIRPORT,OSLO (GARDERMOEN),NO,matched,icao_code,ENZV → ENGM,ENZV|ENGM,True
9,EKCH,ESSA,5574,54.416667,263,22,32,55.617901,12.656000,COPENHAGEN KASTRUP AIRPORT,...,59.648490,17.928829,STOCKHOLM-ARLANDA AIRPORT,STOCKHOLM,SE,matched,icao_code,EKCH → ESSA,EKCH|ESSA,True


## 9. Create airport and monthly summaries

Airport activity counts departures and arrivals separately. A flight from airport A to airport B contributes one departure to A and one arrival to B.

Monthly summaries use `flight_month`, derived from DOF with first-seen fallback. Rows with no usable flight date are retained in the flight table but omitted from month-based summaries and counted in the report.

In [10]:
# Build a directional route key before calculating any summaries.
flights["route_key"] = (
    flights["adep"].astype("string").fillna("<MISSING>")
    + "|"
    + flights["ades"].astype("string").fillna("<MISSING>")
)

departures = (
    flights.dropna(subset=["adep"])
    .groupby("adep")["id"].nunique()
    .rename("departures")
)
arrivals = (
    flights.dropna(subset=["ades"])
    .groupby("ades")["id"].nunique()
    .rename("arrivals")
)
airport_summary = pd.concat([departures, arrivals], axis=1).fillna(0).reset_index()
# The index name can differ depending on pandas version; normalise it explicitly.
airport_summary = airport_summary.rename(columns={airport_summary.columns[0]: "airport_code"})
airport_summary["departures"] = airport_summary["departures"].astype("int64")
airport_summary["arrivals"] = airport_summary["arrivals"].astype("int64")
airport_summary["total_movements"] = airport_summary["departures"] + airport_summary["arrivals"]

airport_summary = airport_summary.merge(
    airport_match[["airport_code", "latitude_deg", "longitude_deg", "airport_name",
                   "municipality", "iso_country", "match_status"]],
    on="airport_code", how="left", validate="one_to_one"
)
airport_summary = airport_summary.sort_values("total_movements", ascending=False).reset_index(drop=True)

dated_flights = flights.dropna(subset=["flight_month"]).copy()
monthly_agg = {"flights": ("id", "nunique"), "directed_route_pairs": ("route_key", "nunique")}
if "icao_operator" in dated_flights.columns:
    monthly_agg["operators"] = ("icao_operator", "nunique")
if "icao24" in dated_flights.columns:
    monthly_agg["aircraft"] = ("icao24", "nunique")

monthly_summary = (
    dated_flights.groupby("flight_month")
    .agg(**monthly_agg)
    .reset_index()
    .sort_values("flight_month")
)

display(airport_summary.head(20))
display(monthly_summary)


,airport_code,departures,arrivals,total_movements,latitude_deg,longitude_deg,airport_name,municipality,iso_country,match_status
0,EHAM,243197,244094,487291,52.308601,4.763890,AMSTERDAM AIRPORT SCHIPHOL,AMSTERDAM,NL,matched
1,EGLL,235697,233416,469113,51.470748,-0.459909,LONDON HEATHROW AIRPORT,LONDON,GB,matched
2,EDDF,213079,216096,429175,50.026706,8.558350,FRANKFURT MAIN AIRPORT,FRANKFURT AM MAIN,DE,matched
3,LTFM,202061,196979,399040,41.274874,28.732136,İSTANBUL AIRPORT,ISTANBUL,TR,matched
4,LFPG,232396,163976,396372,49.008960,2.554117,CHARLES DE GAULLE INTERNATIONAL AIRPORT,"PARIS (ROISSY-EN-FRANCE, VAL-D'OISE)",FR,matched
5,LEMD,186492,208250,394742,40.493407,-3.572249,ADOLFO SUÁREZ MADRID–BARAJAS AIRPORT,MADRID,ES,matched
6,LEBL,183597,183543,367140,41.297100,2.078460,JOSEP TARRADELLAS BARCELONA-EL PRAT AIRPORT,BARCELONA,ES,matched
7,EDDM,168330,169625,337955,48.353802,11.786100,MUNICH AIRPORT,MUNICH,DE,matched
8,LIRF,157534,155760,313294,41.804532,12.251998,ROME–FIUMICINO LEONARDO DA VINCI INTERNATIONAL...,ROME,IT,matched
9,LSZH,132806,132937,265743,47.458056,8.548056,ZÜRICH AIRPORT,ZURICH,CH,matched


,flight_month,flights,directed_route_pairs,operators,aircraft
0,2025-01,1035898,25201,757,28314
1,2025-02,1066341,26426,757,28934
2,2025-03,1235632,30365,762,31524
3,2025-04,1351975,33741,773,32748
4,2025-05,1518167,37600,791,34409
5,2025-06,1574471,40576,784,34867
6,2025-07,1662355,40023,789,35047
7,2025-08,1573717,38278,780,34874
8,2025-09,1535719,37914,782,34525
9,2025-10,1408966,33600,769,33308


## 10. Validation checks and reconciliation

The checks below are designed to catch common processing mistakes before Streamlit consumes the outputs.

- Accepted flight IDs must be unique.
- Directed route keys must be unique.
- Route-pair flight counts must reconcile to the number of accepted flights with both endpoint codes.
- Route coordinates must fall within valid latitude/longitude bounds.
- Every mapped route must have all four endpoint coordinates.
- No negative or zero observation spans are retained.
- Airport summaries must reconcile with departure/arrival totals.

If a check fails, the notebook stops before exporting the dashboard datasets.

In [11]:
validation_results = []

def record_check(name, passed, detail):
    validation_results.append({
        "check": name,
        "passed": bool(passed),
        "detail": str(detail),
    })
    if not passed:
        raise AssertionError(f"VALIDATION FAILED — {name}: {detail}")

record_check(
    "Accepted flight IDs unique",
    flights["id"].is_unique,
    f"{flights['id'].nunique()} unique IDs across {len(flights)} accepted rows"
)
record_check(
    "Directed route pairs unique",
    route_pairs["route_key"].is_unique,
    f"{route_pairs['route_key'].nunique()} unique route keys across {len(route_pairs)} rows"
)
record_check(
    "Route count reconciliation",
    int(route_pairs["flight_count"].sum()) == len(route_source),
    f"route-pair total={int(route_pairs['flight_count'].sum())}; eligible flights={len(route_source)}"
)
record_check(
    "No self-routes",
    not (route_pairs["adep"] == route_pairs["ades"]).any(),
    f"self-route pairs={(route_pairs['adep'] == route_pairs['ades']).sum()}"
)
record_check(
    "Mapped coordinates within geographic bounds",
    (
        route_pairs.loc[route_pairs["is_mapped"], "origin_latitude"].between(-90, 90).all()
        and route_pairs.loc[route_pairs["is_mapped"], "destination_latitude"].between(-90, 90).all()
        and route_pairs.loc[route_pairs["is_mapped"], "origin_longitude"].between(-180, 180).all()
        and route_pairs.loc[route_pairs["is_mapped"], "destination_longitude"].between(-180, 180).all()
    ),
    f"{int(route_pairs['is_mapped'].sum())} mapped route pairs checked"
)
record_check(
    "Observation spans non-negative",
    not (flights["observation_span_min"].dropna() < 0).any(),
    f"{int(flights['observation_span_min'].notna().sum())} valid spans"
)
record_check(
    "Airport summary movement reconciliation",
    int(airport_summary["departures"].sum()) == int(flights["adep"].notna().sum())
    and int(airport_summary["arrivals"].sum()) == int(flights["ades"].notna().sum()),
    "departure and arrival airport counts reconcile to non-missing endpoint records"
)

validation_df = pd.DataFrame(validation_results)
display(validation_df)
print("All validation checks passed.")


,check,passed,detail
0,Accepted flight IDs unique,True,16264265 unique IDs across 16264265 accepted rows
1,Directed route pairs unique,True,86928 unique route keys across 86928 rows
2,Route count reconciliation,True,route-pair total=5578760; eligible flights=557...
3,No self-routes,True,self-route pairs=0
4,Mapped coordinates within geographic bounds,True,78462 mapped route pairs checked
5,Observation spans non-negative,True,16264265 valid spans
6,Airport summary movement reconciliation,True,departure and arrival airport counts reconcile...


All validation checks passed.


## 11. Build unmatched-airport audit and run-level report

Unmatched airports are not silently discarded. The audit includes the endpoint code and the reason it was not mapped. The run report captures source files, row counts, duplicate handling, route coverage, date coverage, and validation outcomes.

In [12]:
# Endpoint-level unmatched code frequency.
unmatched_origin = (
    flights.loc[flights["origin_match_status"].ne("matched"), "adep"]
    .value_counts(dropna=False).rename_axis("airport_code").reset_index(name="origin_flight_count")
)
unmatched_destination = (
    flights.loc[flights["destination_match_status"].ne("matched"), "ades"]
    .value_counts(dropna=False).rename_axis("airport_code").reset_index(name="destination_flight_count")
)
unmatched_airports = unmatched_codes.merge(
    unmatched_origin, on="airport_code", how="left"
).merge(
    unmatched_destination, on="airport_code", how="left"
)
unmatched_airports[["origin_flight_count", "destination_flight_count"]] = (
    unmatched_airports[["origin_flight_count", "destination_flight_count"]].fillna(0).astype("int64")
)

date_min = flights["flight_datetime_utc"].min()
date_max = flights["flight_datetime_utc"].max()

report = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "input_directory": str(FLIGHT_DIR.resolve()),
    "airport_reference_file": str(AIRPORTS_FILE.resolve()),
    "output_directory": str(OUTPUT_DIR.resolve()),
    "source_files": [p.name for p in source_files],
    "source_file_count": len(source_files),
    "distinct_filename_months": int(distinct_filename_months),
    "source_rows_after_missing_id_removal": int(len(all_raw_standardised)),
    "exact_duplicate_rows_removed": int(exact_duplicates_removed),
    "conflicting_flight_ids_quarantined": int(len(conflict_id_set)),
    "accepted_flight_rows": int(len(flights)),
    "accepted_unique_flight_ids": int(flights["id"].nunique()),
    "flights_with_both_route_endpoints": int(len(route_source)),
    "unique_directed_route_pairs": int(len(route_pairs)),
    "mapped_directed_route_pairs": int(route_pairs["is_mapped"].sum()),
    "unmapped_directed_route_pairs": int((~route_pairs["is_mapped"]).sum()),
    "unique_airport_codes_in_flights": int(len(airport_match)),
    "unmatched_or_ambiguous_airport_codes": int(len(unmatched_codes)),
    "flights_without_flight_date": int(flights["flight_month"].isna().sum()),
    "flight_date_min_utc": None if pd.isna(date_min) else date_min.isoformat(),
    "flight_date_max_utc": None if pd.isna(date_max) else date_max.isoformat(),
    "validation_checks": validation_results,
    "notes": [
        "Route pairs are directional ADEP-to-ADES combinations.",
        "Conflicting duplicate IDs are quarantined, not arbitrarily resolved.",
        "Airport coordinates are airport locations, not actual flown trajectories.",
        "first_seen-to-last_seen is an observation-span proxy, not confirmed block time.",
    ],
}
print(json.dumps(report, indent=2, default=str)[:12000])


{
  "run_timestamp_utc": "2026-11-05T23:30:42.173108+00:00",
  "input_directory": "C:\\Users\\Toni Manjani\\Desktop\\airline_flight_list_dashboard_full_year_map_fixed\\airline_flight_list_dashboard\\data\\flight_list",
  "airport_reference_file": "C:\\Users\\Toni Manjani\\Desktop\\airline_flight_list_dashboard_full_year_map_fixed\\airline_flight_list_dashboard\\data\\airports.csv",
  "output_directory": "C:\\Users\\Toni Manjani\\Desktop\\airline_flight_list_dashboard_full_year_map_fixed\\airline_flight_list_dashboard\\processed_data",
  "source_files": [
    "flight_list_202501.parquet",
    "flight_list_202502.parquet",
    "flight_list_202503.parquet",
    "flight_list_202504.parquet",
    "flight_list_202505.parquet",
    "flight_list_202506.parquet",
    "flight_list_202507.parquet",
    "flight_list_202508.parquet",
    "flight_list_202509.parquet",
    "flight_list_202510.parquet",
    "flight_list_202511.parquet",
    "flight_list_202512.parquet"
  ],
  "source_file_count": 12,


## 12. Export the processed datasets

Parquet is used for the main datasets because it is compact and fast to load from pandas/Streamlit. CSV files are also exported for the audit tables so they can be opened directly in Excel.

The export step writes to a dedicated output folder and does not overwrite the raw monthly inputs.

In [14]:
# 12. Export the processed datasets

# Convert object columns to consistent strings before Parquet export.
# This prevents ArrowTypeError when a column contains mixed values,
# such as numeric timestamps in some rows and strings in others.

for filename, frame in outputs.items():

    object_cols = frame.select_dtypes(include=["object"]).columns

    for col in object_cols:
        frame[col] = frame[col].astype("string")

    target = OUTPUT_DIR / filename

    frame.to_parquet(
        target,
        index=False,
        compression="snappy"
    )

    print(
        f"Wrote {target.name}: "
        f"{len(frame):,} rows, "
        f"{target.stat().st_size / (1024**2):.2f} MB"
    )


# Export audit tables
duplicate_conflicts.to_csv(
    OUTPUT_DIR / "duplicate_conflicts.csv",
    index=False
)

unmatched_airports.to_csv(
    OUTPUT_DIR / "unmatched_airports.csv",
    index=False
)

file_read_audit.to_csv(
    OUTPUT_DIR / "source_file_audit.csv",
    index=False
)

validation_df.to_csv(
    OUTPUT_DIR / "validation_results.csv",
    index=False
)


# Export the processing report
(OUTPUT_DIR / "preprocessing_report.json").write_text(
    json.dumps(report, indent=2, default=str),
    encoding="utf-8"
)

print("\nExport completed successfully!")
print("Output directory:", OUTPUT_DIR.resolve())

Wrote flights_processed.parquet: 16,264,265 rows, 1164.88 MB
Wrote route_pairs.parquet: 86,928 rows, 2.58 MB
Wrote airport_summary.parquet: 1,115 rows, 0.07 MB
Wrote monthly_summary.parquet: 12 rows, 0.00 MB

Export completed successfully!
Output directory: C:\Users\Toni Manjani\Desktop\airline_flight_list_dashboard_full_year_map_fixed\airline_flight_list_dashboard\processed_data


## 13. Re-open the exported files and perform a round-trip check

A successful write is not enough. Re-open each Parquet file and verify that row counts and route-pair keys remain intact. This catches serialization problems before Streamlit reads the files.

In [15]:
round_trip = {}
for filename, expected in outputs.items():
    loaded = pd.read_parquet(OUTPUT_DIR / filename)
    round_trip[filename] = {
        "expected_rows": len(expected),
        "reloaded_rows": len(loaded),
        "row_count_matches": len(expected) == len(loaded),
    }
    assert len(loaded) == len(expected), f"Round-trip row count mismatch: {filename}"

reloaded_routes = pd.read_parquet(OUTPUT_DIR / "route_pairs.parquet")
assert reloaded_routes["route_key"].is_unique
assert int(reloaded_routes["flight_count"].sum()) == len(route_source)

display(pd.DataFrame(round_trip).T)
print("Parquet round-trip checks passed.")


,expected_rows,reloaded_rows,row_count_matches
flights_processed.parquet,16264265,16264265,True
route_pairs.parquet,86928,86928,True
airport_summary.parquet,1115,1115,True
monthly_summary.parquet,12,12,True


Parquet round-trip checks passed.


## 14. How Streamlit should consume these outputs

After preprocessing has completed successfully, the dashboard should read the processed outputs instead of loading and merging all monthly raw files at startup.

Example:

```python
from pathlib import Path
import pandas as pd
import streamlit as st

PROCESSED = Path("processed_data")

@st.cache_data
def load_dashboard_data():
    flights = pd.read_parquet(PROCESSED / "flights_processed.parquet")
    routes = pd.read_parquet(PROCESSED / "route_pairs.parquet")
    airports = pd.read_parquet(PROCESSED / "airport_summary.parquet")
    monthly = pd.read_parquet(PROCESSED / "monthly_summary.parquet")
    return flights, routes, airports, monthly

flights, routes, airports, monthly = load_dashboard_data()
```

For the network map, use `route_pairs.parquet` and filter to `is_mapped == True`. Use the `origin_latitude`, `origin_longitude`, `destination_latitude`, and `destination_longitude` columns. For the flight explorer, use `flights_processed.parquet`.

### Refresh procedure

1. Add the new monthly Parquet file(s) to `data/flight_list/`.
2. Run this notebook from top to bottom.
3. Review the validation table, `duplicate_conflicts.csv`, and `unmatched_airports.csv`.
4. Only after all checks pass, restart or refresh Streamlit.

### Metric caveats

- `flight_count` counts accepted unique flight IDs, not passengers or seats.
- `unique_aircraft` counts distinct `icao24` values when available; missing aircraft identifiers are not counted as aircraft.
- `first_seen`–`last_seen` is an observation span and must not be labelled actual block time without independent validation.
- Airport route links are network relationships, not geodesic or radar-track reconstructions.

## 15. Final processing checklist

Before treating the output as a full-year dataset, confirm:

- [ ] All 12 monthly Parquet files are listed in the inventory.
- [ ] The inferred month coverage is correct and no month is missing.
- [ ] `duplicate_conflicts.csv` has been reviewed (ideally zero conflicts, or conflicts resolved using a documented source rule).
- [ ] `unmatched_airports.csv` has been reviewed, especially codes contributing many flights.
- [ ] All validation checks pass.
- [ ] Parquet round-trip checks pass.
- [ ] Streamlit points to `processed_data/`, not the raw monthly input folder.

**A partial set of source files can still be processed, but it must not be described as a full-year result.**